In [73]:
import pandas as pd
import numpy as np
import re
# Set display options to show full content in columns
pd.set_option('display.max_colwidth', None)
import sys
from pathlib import Path

# Get the path to the current script
current_dir = Path.cwd()

# Go one level up
current_dir = current_dir.parent

# Add the 'scripts' directory to sys.path to be able to import data_utils.py
sys.path.append(str(current_dir))

from scripts.data_utils import split_summary_methods, format_mean_std
import ast  
import matplotlib.pyplot as plt
import seaborn as sns
from autorank import autorank, plot_stats, create_report
pd.set_option('display.max_colwidth', None)

In [74]:
MAX_NUMBER_OF_RUNS_STOCASTIC = 3
MAX_SCENARIOS = 2 #20 for Evaluation or 2 Tuning
TYPE_WS = "params_with_p_window_size" #params_with_p_window_size or params_with_window

data = pd.read_excel(current_dir / "datasets" / "summaries" / "summary_results_sota_and_own_method_tuning_pds_and_pv.xlsx")

data['method_window_and_param'] = data['method_window_and_param'].str.replace("SWKNN_own", "SWKNN(own)")
data['code_and_name_scenario'] = data['cod_scenario'] +"_"+ data['name_scenario'].astype(str)

In [75]:
# Find duplicates for the same iteration and the same experiment
duplicates = data[["iteration", "cod_scenario", "method_window_and_param", "count_cleaned_score"]].duplicated(keep=False)
data[duplicates][["iteration", "cod_scenario", "method_window_and_param", "count_cleaned_score"]].sort_values(by=["method_window_and_param"])

,iteration,cod_scenario,method_window_and_param,count_cleaned_score


In [76]:
# Summary of datasets used
pivot = data.pivot_table(values=["count_cleaned_score", "count_anomalies"], index=["cod_scenario"], aggfunc='mean')
pivot['percentage_anomalies'] = (pivot['count_anomalies'] / pivot['count_cleaned_score']) * 100
pivot


,count_anomalies,count_cleaned_score,percentage_anomalies
cod_scenario,,,
116,492.0,10000.0,4.920000
DA3,600.0,4200.0,14.285714


In [77]:
# Adjusting format and creating new columns for analysis in the dataset read

# Extracting new columns for the method, window_size, parameters
data[['method',"window_size", 'parameters']] = data['method_window_and_param'].apply(
    lambda x: pd.Series(split_summary_methods(x))
)

# Creating new column params_with_window
data["params_with_p_window_size"] = "{'p_window_size': " + data["p_window_size"].astype(str)+data["parameters"].str.replace("{", ", ", regex=False)
data["params_with_p_window_size"] = data["params_with_p_window_size"].str.replace("}_{", ",", regex=False)

data["params_with_window"] = "{'window_size': " + data["window_size"].astype(str)+data["parameters"].str.replace("{", ", ", regex=False)
data["params_with_window"] = data["params_with_window"].str.replace("}_{", ",", regex=False)

# Define your list of non-stochastic methods
non_stochastic_methods = ["SWKNN", "SWKNN_own", "SWLOF", "KitNet", "ExactStorm"]

# Create the 'stochastic' column
data['stochastic'] = data['method'].apply(
    lambda x: 'N' if x in non_stochastic_methods else 'Y')

# Shortening the name of the methods
new_method_name = {"xStream":"XStream", "RSHash":"RSHash", "IForestASD":"IFASD", "RobustRandomCutForest":"RRCF", "KitNet":"KitNet", "ExactStorm":"EStorm","oIF":"OIF", "HStree":"HStree", "OnlineBootKNN":"OBKNN"} 
data["method"] = data['method'].replace(new_method_name)
data["method_with_param"] = data['method']+"_"+data["parameters"]

#Separating OnlineBootKNN with None, Z-normalization or SQRT transformation
# Case 1: None
mask_none = (data['method'] == 'OBKNN') & \
           (data['params_with_window'].str.contains('NONE', case=False, na=False))
data.loc[mask_none, 'method'] = 'OBKNN (TNone)'

# Case 2: ZNormalization
mask_znorm = (data['method'] == 'OBKNN') & \
           (data['params_with_window'].str.contains('ZNORM', case=False, na=False))
data.loc[mask_znorm, 'method'] = 'OBKNN (TZnorm)'

# Case 2: ZNormalization
mask_sqrt = (data['method'] == 'OBKNN') & \
           (data['params_with_window'].str.contains('SQRT', case=False, na=False))
data.loc[mask_sqrt, 'method'] = 'OBKNN (TSqrt)'

# Including the year of publication of the methods
method_and_year = {"XStream":"2018", "RSHash":"2011", "IFASD":"2013", "RRCF":"2016", "KitNet":"2018", "EStorm":"2007","OIF":"2024", "HStree":"2011", "OBKNN (TNone)":"2025", "OBKNN (TZnorm)":"2025", "SWKNN":"2000","SWLOF":"2000"} 
data["method_and_year"] = data['method'].map(lambda x: f"{x} ({method_and_year.get(x, 'Unknown')})")





In [78]:
# Obtaining the hyperparameters that are complete for analysis
averaged_results = data.groupby(['method', 'stochastic', TYPE_WS, 'cod_scenario'])['raw_pr_auc'].agg(
    n_runs='count',
).reset_index()

# Condition 1: Stochastic methods must have MAX_NUMBER_OF_RUNS_STOCASTIC
cond_stochastic = (averaged_results['n_runs'] == MAX_NUMBER_OF_RUNS_STOCASTIC) & (averaged_results['stochastic'] == 'Y')

# Condition 2: Non-Stochastic methods must have 1 run
cond_non_stochastic = (averaged_results['n_runs'] == 1) & (averaged_results['stochastic'] == 'N')

# Apply the combined filter using | (OR)
# Let's call this new variable 'filtered_results'
filtered_results = averaged_results[cond_stochastic | cond_non_stochastic]

# Now, 'filtered_results' contains the data you wanted to see
# You can uncomment the next line to inspect it:
# print("--- Filtered Results (Step 1) ---")
# display(filtered_results)


# Now, aggregate the *filtered* results to count complete scenarios
scenario_counts = filtered_results.groupby(['method', TYPE_WS])['cod_scenario'].agg(
    n_scenarios='nunique'
).reset_index()

# Get the final list of complete hyperparameters
scenario_counts = scenario_counts[scenario_counts.n_scenarios==MAX_SCENARIOS]
hyperparameters_complete = scenario_counts[TYPE_WS]


In [79]:
#Applying filters to analyse data

regex_filter = "|".join(["mDragStream"])#Including under work method

filtered_data = data[~data['method_window_and_param'].str.contains(regex_filter, regex=True)]

filtered_data = filtered_data.sort_values(by='cod_scenario')

filtered_data = filtered_data[filtered_data[TYPE_WS].isin(hyperparameters_complete)]


In [80]:
# Summary of datasets used (after data filtering)
pivot = filtered_data.pivot_table(values=["count_cleaned_score", "count_anomalies"], index=["cod_scenario"], aggfunc='mean')

pivot['percentage_anomalies'] = (pivot['count_anomalies'] / pivot['count_cleaned_score']) * 100
pivot

,count_anomalies,count_cleaned_score,percentage_anomalies
cod_scenario,,,
116,492.0,10000.0,4.920000
DA3,600.0,4200.0,14.285714


In [81]:
# Obtaining the best hyperparameters
# 1. average results over all iterations and all scenarios
averaged_results = filtered_data.groupby(['method', TYPE_WS, 'cod_scenario'])['raw_pr_auc'].agg(
    mean_auc_pr='mean',
    std_auc_pr='std',
    n_runs='count'
).reset_index()

# 2. average results for each method and hyperparameter for all scenarios
summary = averaged_results.groupby(['method', TYPE_WS]).agg(
    mean_auc_pr=('mean_auc_pr', 'mean'),
    beetween_std_auc_pr=('mean_auc_pr', 'std'),
    within_std_auc_pr=('std_auc_pr', 'mean'),
    n_scenarios=('mean_auc_pr', 'count'),
    n_experiments=('n_runs', 'sum')
).reset_index()


# Obtaining the best hyperparameters 
# 3. Select the highest AUC_PR per method and hyperparameter with lowest standard deviation beetween scenarios
best_results = (
    summary
    .sort_values(['mean_auc_pr','beetween_std_auc_pr'], ascending=[False, True])
    .groupby('method')
    .head(1)
)
best_params = best_results[TYPE_WS].unique().tolist()
best_results

,method,params_with_p_window_size,mean_auc_pr,beetween_std_auc_pr,within_std_auc_pr,n_scenarios,n_experiments
102,OBKNN (TNone),"{'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}",0.967986,0.043575,0.003943,2,6
41,IFASD,"{'p_window_size': 0.2, 'initial_window_X': None}",0.959322,0.044329,0.003482,2,6
54,KitNet,"{'p_window_size': 0.05, 'hidden_ratio': 0.9, 'learning_rate': 0.1, 'max_size_ae': 10}",0.943090,0.032492,NaN,2,2
223,SWKNN,"{'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}",0.936771,0.089419,NaN,2,2
185,OIF,"{'p_window_size': 0.2, 'growth_criterion': 'fixed', 'max_leaf_samples': 64, 'n_jobs': -1, 'num_trees': 64}",0.648609,0.496942,0.020312,2,6
14,EStorm,"{'p_window_size': 0.2, 'max_radius': 900}",0.557143,0.045457,NaN,2,2
153,OBKNN (TZnorm),"{'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}",0.548261,0.625520,0.000421,2,6
20,HStree,"{'p_window_size': 0.02, 'anomaly_threshold': 0.5, 'max_depth': 10, 'number_of_trees': 25, 'size_limit': 0.1}",0.480364,0.644116,0.007364,2,6
248,XStream,"{'p_window_size': 0.02, 'depth': 25, 'n_chains': 100, 'num_components': 50}",0.457042,0.083369,0.021800,2,6
234,SWLOF,"{'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'metric': 'euclidean', 'simplified': False}",0.362281,0.268258,NaN,2,2


## Summary Score of Online Anomaly Detectors with PV Datasets 

In [82]:
# Filtering just the best hyperparameters
filtered_data = filtered_data[filtered_data[TYPE_WS].isin(best_params)]

filtered_data.params_with_p_window_size.unique()

array(["{'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'max_node_size': 20, 'metric': 'cityblock', 'min_node_size': 5, 'split_sampling': 5}",
       "{'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'NONE', 'type_dist': 'largest', 'update_distance_with_abnormal': False, 'update_mode_stats': 'ema'}",
       "{'p_window_size': 0.2, 'algorithm': 'brute', 'alpha_ema': 0.01, 'alpha_z_test': 0.05, 'chunk_size': 30, 'dmetric': 'cityblock', 'ensemble_size': 30, 'n_jobs': -1, 'no_bootstrapp': False, 'no_z_score': False, 'transf': 'ZNORM', 'type_dist': 'largest', 'update_distance_with_abnormal': True, 'update_mode_stats': 'welford'}",
       "{'p_window_size': 0.02, 'depth': 25, 'n_chains': 100, 'num_components': 50}",
       "{'p_window_size': 0.02, 'k': 10, 'k_is_max': False, 'metric': 'euclidean', 'simplified': False}",
 

In [83]:

# 1. Pivot for mean - window_size is excluded to aggregate data per scenario
mean_pivot = filtered_data.pivot_table(
    values="raw_pr_auc",
    columns=["code_and_name_scenario"],
    index=["method"],
    aggfunc="mean"
)

# Pivot for std
std_pivot = filtered_data.pivot_table(
    values="raw_pr_auc",
    columns=["code_and_name_scenario"],
    index=["method"],
    aggfunc="std"
)

# Ensure column alignment
std_pivot = std_pivot.reindex(columns=mean_pivot.columns)

# 2. Stack the DataFrames
# Only one level exists in columns now
mean_s = mean_pivot.stack(dropna=False)
std_s = std_pivot.stack(dropna=False)

# 3. Apply the formatting function
combined_s = pd.DataFrame({'mean': mean_s, 'std': std_s}).apply(
    lambda row: format_mean_std(row['mean'], row['std']), 
    axis=1
)

# 4. Unstack back into a DataFrame
combined = combined_s.unstack()

# 5. Get numeric averages for the final column and row sorting
avg_mean = mean_pivot.mean(axis=1)
avg_std = std_pivot.mean(axis=1)

avg_col_s = pd.DataFrame({'mean': avg_mean, 'std': avg_std}).apply(
    lambda row: format_mean_std(row['mean'], row['std']),
    axis=1
)

combined["Avg"] = avg_col_s

# 6. Sort rows by the numeric average mean descending
combined = combined.loc[avg_mean.sort_values(ascending=False).index]

combined

/tmp/ipykernel_1439621/1971578523.py:22: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  mean_s = mean_pivot.stack(dropna=False)
/tmp/ipykernel_1439621/1971578523.py:23: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  std_s = std_pivot.stack(dropna=False)


code_and_name_scenario,116_TAO,DA3_20250610,Avg
method,,,
OBKNN (TNone),0.937 ± 7.9e-03,0.999 ± 1.1e-05,0.968 ± 3.9e-03
IFASD,0.991 ± 4.9e-03,0.928 ± 2.1e-03,0.959 ± 3.5e-03
KitNet,0.966 ± NA,0.920 ± NA,0.943 ± NA
SWKNN,1.000 ± NA,0.874 ± NA,0.937 ± NA
OIF,1.000 ± 0.0e+00,0.297 ± 4.1e-02,0.649 ± 2.0e-02
EStorm,0.525 ± NA,0.589 ± NA,0.557 ± NA
OBKNN (TZnorm),0.991 ± 5.8e-04,0.106 ± 2.6e-04,0.548 ± 4.2e-04
HStree,0.025 ± 2.9e-06,0.936 ± 1.5e-02,0.480 ± 7.4e-03
XStream,0.516 ± 2.0e-02,0.398 ± 2.4e-02,0.457 ± 2.2e-02


In [84]:

dataset_summary = filtered_data.groupby('code_and_name_scenario').agg(
    total_normal=('count_normal', 'max'),
    abn_instances=('count_anomalies', 'max')
).reset_index()

dataset_summary['total_instances'] = dataset_summary['abn_instances'] + dataset_summary['total_normal']
dataset_summary['anomaly_percentage'] = (dataset_summary['abn_instances'] / dataset_summary['total_instances']) * 100

dataset_info = {
    'TAO': {'anomaly_type_len': 'Point&Seq.', 'features': 3},
    '20250610': {'anomaly_type_len': 'Spatial', 'features': 2048}
}

ordered_list = list(dataset_info.keys())

dataset_summary['matched_dataset'] = dataset_summary['code_and_name_scenario'].str.split('_').str[-1]
dataset_summary['matched_dataset'] = dataset_summary['matched_dataset'].replace({'OPPORTUNITY': 'OPP'})

for col in ['anomaly_type_len', 'features']:
    dataset_summary[col] = dataset_summary['matched_dataset'].map(
        lambda x: dataset_info[x][col] if x in dataset_info else pd.NA
    )

dataset_summary['matched_dataset'] = pd.Categorical(
    dataset_summary['matched_dataset'], 
    categories=ordered_list, 
    ordered=True
)

dataset_summary = dataset_summary.dropna(subset=['matched_dataset'])
dataset_summary = dataset_summary.sort_values('matched_dataset')

dataset_summary = dataset_summary[
    ['code_and_name_scenario', 'features', 'abn_instances', 'total_instances', 'anomaly_percentage']
].reset_index(drop=True)

dataset_summary

,code_and_name_scenario,features,abn_instances,total_instances,anomaly_percentage
0,116_TAO,3,492,10000,4.920000
1,DA3_20250610,2048,600,4200,14.285714
